Name: Rucha Vikrant Wadhavankar

SRN: PES2UG23CS496

In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 9.9 MB/s eta 0:00:00


#CoT

In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

Enter your Groq API Key: ··········


In [3]:
question = "Ram has 9 cricket balls. He buys 2 more packets of cricket balls. Each can has 6 cricket balls. How many does he have now?"

prompt_standard = f"Answer this question: {question}"
print("--- STANDARD (Llama3.1-8b) ---")
print(llm.invoke(prompt_standard).content)

--- STANDARD (Llama3.1-8b) ---
To find out how many cricket balls Ram has now, we need to calculate the total number of cricket balls he has after buying 2 more packets.

Ram initially has 9 cricket balls. Each packet has 6 cricket balls, and he buys 2 packets. 

So, the total number of cricket balls in the 2 packets is 2 * 6 = 12.

Adding this to the initial 9 cricket balls, Ram now has 9 + 12 = 21 cricket balls.

Therefore, Ram has 21 cricket balls now.


In [4]:
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- Chain of Thought (Llama3.1-8b) ---")
print(llm.invoke(prompt_cot).content)

--- Chain of Thought (Llama3.1-8b) ---
To find out how many cricket balls Ram has now, we need to follow these steps:

1. Ram already has 9 cricket balls.
2. He buys 2 more packets of cricket balls. Since each packet has 6 cricket balls, we need to find out how many cricket balls are in 2 packets.
   2 packets * 6 cricket balls per packet = 12 cricket balls
3. Now, we add the cricket balls Ram already had to the new cricket balls he bought.
   9 (initial cricket balls) + 12 (new cricket balls) = 21

So, Ram now has 21 cricket balls.


#ToT

In [5]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get my 3-year-old to eat vegetables?"

prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'

    1: {sol1}
    2: {sol2}
    3: {sol3}

    Act as a Child Psychologist. Pick the most sustainable one (not bribery) and explain why.
    """
)

tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
As a Child Psychologist, I would recommend Solution 2 from the second set of solutions: "Sneaky" Veggie Smoothies. This approach is sustainable for several reasons:

1. **Gradual exposure**: By blending finely chopped vegetables with their favorite fruits, you're allowing your child to gradually get used to the taste and texture of vegetables. This gradual exposure can help them become more comfortable with the idea of eating vegetables.
2. **Minimizes resistance**: Since the vegetables are blended into a smoothie, your child is less likely to resist or reject them. This reduces the likelihood of power struggles and mealtime conflicts.
3. **Encourages healthy habits**: By incorporating vegetables into a smoothie, you're teaching your child the importance of including a variety of nutrients in their diet. This sets a positive example and encourages healthy eating habits.
4. **Flexibility**: You can adjust the amount of vegetables in the smoothie bas

#GoT

In [7]:
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 2-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Comedy") | llm | StrOutputParser(),
)

prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Comedy: {draft_horror}

    Your task: Create a new Mega-Movie that combines the TECHNOLOGY of Sci-Fi, the PASSION of Romance, and the FUN of Comedy.
    Write one paragraph.
    """
)

got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
print(got_chain.invoke("Time Travel"))

--- Graph of Thoughts (GoT) Result ---
Introducing "Chrono Crush": A Time-Traveling Romantic Comedy. When brilliant physicist and struggling artist Sophie discovers a cutting-edge time machine disguised as a vintage toaster, she sees it as the perfect opportunity to travel back in time and meet her childhood idol, a charismatic artist who inspired her to pursue her passion. However, things take a surprising turn when Sophie meets not only her idol but also a charming stranger who has just discovered his own time-traveling abilities. As they navigate through different eras, from ancient Rome to the 1920s jazz scene, Sophie finds herself caught in a whirlwind of romance and adventure, but must also confront the consequences of altering the timeline and the possibility that her actions may have a profound impact on her own family and loved ones.
